In [14]:
import os
import requests
import gradio as gr
from typing import Annotated, TypedDict, List, Dict, Any
from dotenv import load_dotenv

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages

load_dotenv()

True

In [15]:

SPORTS_API_KEY = os.getenv("SPORTS_API_KEY")
OPENAI_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [5]:
BASE_URL = "https://v3.football.api-sports.io"

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0
)

# Tools

In [16]:
@tool
def get_recent_matches(team: str) -> str:
    """Get last 5 matches of a football team"""

    url = f"{BASE_URL}/teams"
    headers = {"x-apisports-key": SPORTS_API_KEY}
    params = {"search": team}

    r = requests.get(url, headers=headers, params=params).json()

    if not r["response"]:
        return "Team not found."

    team_id = r["response"][0]["team"]["id"]

    url = f"{BASE_URL}/fixtures"
    params = {"team": team_id, "last": 5}

    r = requests.get(url, headers=headers, params=params).json()

    matches = []
    matches = []

    for m in r["response"]:
        home = m["teams"]["home"]["name"]
        away = m["teams"]["away"]["name"]
        goals_home = m["goals"]["home"]
        goals_away = m["goals"]["away"]

        matches.append(
            f"{home} {goals_home} - {goals_away} {away}"
        )

    return "\n".join(matches)


tools = [get_recent_matches]

tool_node = ToolNode(tools)

llm_with_tools = llm.bind_tools(tools)

# Agent State

In [18]:
class AgentState(TypedDict):
    messages: Annotated[List, add_messages]


## The Planner Node

In [19]:
def planner(state: AgentState):
    messages = state["messages"]

    system_prompt = SystemMessage(
        content="""
You are an AI Sports Analyst.

You analyze football teams using tools.
Always fetch recent matches before analysis.
Provide insights about team form and performance.
"""
    )

    response = llm_with_tools.invoke([system_prompt] + messages)

    return {"messages": [response]}

## Analyzer Node

In [20]:
def analyzer(state: AgentState):
    messages = state["messages"]

    analysis_prompt = SystemMessage(
        content="""
Analyze the match results and summarize:
- Wins
- Losses
- Form trend
- Insights
"""
    )

    response = llm.invoke([analysis_prompt] + messages)

    return {"messages": [response]}

## Evaluator Node

In [21]:
def evaluator(state: AgentState):
    messages = state["messages"]

    evaluation_prompt = SystemMessage(
        content="""
Evaluate if the analysis is clear and helpful.
If not, improve it.
"""
    )

    response = llm.invoke([evaluation_prompt] + messages)

    return {"messages": [response]}

## Build Gragph

In [22]:
graph = StateGraph(AgentState)

graph.add_node("planner", planner)
graph.add_node("tools", tool_node)
graph.add_node("analyzer", analyzer)
graph.add_node("evaluator", evaluator)

graph.add_edge(START, "planner")
graph.add_edge("planner", "tools")
graph.add_edge("tools", "analyzer")
graph.add_edge("analyzer", "evaluator")
graph.add_edge("evaluator", END)

memory = MemorySaver()

app = graph.compile(checkpointer=memory)


# Gradio

In [25]:
# -------------------------
# Gradio Functions
# -------------------------
def process_message(message, chat_history):
    """
    Invokes the LangGraph autonomous agent and updates the chat history.
    """
    result = app.invoke(
        {"messages": [HumanMessage(content=message)]},
        config={"configurable": {"thread_id": "sports-agent"}}
    )

    response = result["messages"][-1].content

    # Append assistant response to chat history
    chat_history.append({"role": "assistant", "content": response})

    return chat_history


def reset_chat():
    """Clears the chat history"""
    return []

In [26]:
with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## ⚽ SportSense AI — Autonomous Sports Analyst")

    chatbot = gr.Chatbot(height=350, type="messages")

    message = gr.Textbox(
        placeholder="Ask about a team (e.g., Arsenal recent form)"
    )

    # Buttons
    with gr.Row():
        submit_button = gr.Button("Go!", variant="primary")
        clear_button = gr.Button("Reset", variant="stop")

    # Function to process message
    def process_message_click(msg, chat_history):
        return process_message(msg, chat_history)

    # Function to reset chat
    def reset_chat():
        return []

    # Bind events
    submit_button.click(
        process_message_click,
        [message, chatbot],
        [chatbot]
    )

    clear_button.click(
        reset_chat,
        [],
        [chatbot]
    )

    # Also allow pressing Enter to submit
    message.submit(process_message_click, [message, chatbot], [chatbot])

demo.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
